In [30]:
import pandas as pd
import matplotlib.pyplot as plt
import re

In [31]:
pd.set_option('max_colwidth', 800)

## Initial data load & cleaning

In [11]:
data_load = pd.read_csv('../trash_hauler_report.csv')

In [12]:
data_load

,Request Number,Date Opened,Request,Description,Incident Address,Zip Code,Trash Hauler,Trash Route,Council District,State Plan X,State Plan Y
0,25270,11/1/2017,Trash - Backdoor,"house with the wheel chair ramp, they share dr...",3817 Crouch Dr,37207.0,RED RIVER,3205,2.0,1727970.412,686779.4781
1,25274,11/1/2017,Trash - Curbside/Alley Missed Pickup,Curb/Trash miss Tuesday.,4028 Clarksville Pike,37218.0,RED RIVER,4202,1.0,1721259.366,685444.7996
2,25276,11/1/2017,Trash - Curbside/Alley Missed Pickup,Curb/trash miss Tuesday.,6528 Thunderbird Dr,37209.0,RED RIVER,4205,20.0,1707026.753,659887.4716
3,25307,11/1/2017,Trash - Curbside/Alley Missed Pickup,missed,2603 old matthews rd,37207.0,WASTE IND,2206,2.0,1735691.771,685027.2459
4,25312,11/1/2017,Trash - Curbside/Alley Missed Pickup,Missed the even side of the road.,604 croley dr,37209.0,RED RIVER,4203,20.0,1710185.772,664205.1011
...,...,...,...,...,...,...,...,...,...,...,...
20221,267125,11/1/2019,Trash - Curbside/Alley Missed Pickup,MISSED...NEIGHBORS MISSED,2731 Murfreesboro Pike,37013.0,RED RIVER,4502,32.0,1781137.263,632448.5511
20222,267126,11/1/2019,Trash - Curbside/Alley Missed Pickup,entire alley,"1621 Long Ave, Nashville, TN 37206, United States",37206.0,METRO,9508,6.0,1749711.399,669201.6016
20223,267130,11/1/2019,Trash - Curbside/Alley Missed Pickup,missed several,"2943 Windemere Cir, Nashville, TN 37214, Unite...",37214.0,RED RIVER,1502,15.0,1770293.388,674936.3038
20224,267134,11/1/2019,Trash - Curbside/Alley Missed Pickup,Caller stated trash was missed & were only pic...,"3325 Murfreesboro Pike, Nashville, TN 37013, U...",37013.0,RED RIVER,4502,32.0,1785224.998,627146.4002


In [13]:
data_load.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20226 entries, 0 to 20225
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Request Number    20226 non-null  int64  
 1   Date Opened       20226 non-null  object 
 2   Request           20226 non-null  object 
 3   Description       20195 non-null  object 
 4   Incident Address  20217 non-null  object 
 5   Zip Code          20151 non-null  float64
 6   Trash Hauler      19325 non-null  object 
 7   Trash Route       19279 non-null  object 
 8   Council District  20177 non-null  float64
 9   State Plan X      20198 non-null  float64
 10  State Plan Y      20198 non-null  float64
dtypes: float64(4), int64(1), object(6)
memory usage: 1.7+ MB


In [18]:
data_load['Request '].value_counts()

Request 
Trash - Curbside/Alley Missed Pickup    15028
Trash - Backdoor                         2629
Trash Collection Complaint               2312
Damage to Property                        257
Name: count, dtype: int64

#### First I want to find all the rows that refer to missed pickups.

A quick look through the different request types (below) indicates that merely filtering for 'Curbside/Alley Missed Pickup' won't suffice, as there are calls about missed pickups in the 'Backdoor' and 'Trash Collection Complaint' categories as well.  I don't see any under 'Damage to Property', but will look for them just in case.

In [28]:
data_load.loc[data_load['Request '] == 'Trash - Curbside/Alley Missed Pickup', 'Description']

1                                                            Curb/Trash miss Tuesday.
2                                                            Curb/trash miss Tuesday.
3                                                                              missed
4                                                   Missed the even side of the road.
8                                                                             Missed.
                                             ...                                     
20221                                                       MISSED...NEIGHBORS MISSED
20222                                                                    entire alley
20223                                                                  missed several
20224    Caller stated trash was missed & were only picked up 3x in the last 3 month.
20225                                                  possibly others missed as well
Name: Description, Length: 15028, dtype: object

In [26]:
data_load.loc[data_load['Request '] == 'Trash - Backdoor', 'Description']

0        house with the wheel chair ramp, they share driveway, in back driveway. 3817 Crouch Dr near NES light post is. \r\n615-876-6274
55                                                                               Backdoor/miss for last Friday. Does not understand why?
63                                          Missed trash pickup said has been picking up behind the gate for 26 years \r\n(615) 333-0065
71                                                                                                                               Missed.
78                                  Missed- she said her grandson was over yesterday and he told her it was full. She needs it picked up
                                                                      ...                                                               
20165                                                           HAS MISSED BACK DOOR TRASH PICK UP AGAIN/ ALSO MISSED LAST WEEKS PICK UP
20167                                    

In [27]:
data_load.loc[data_load['Request '] == 'Trash Collection Complaint', 'Description']

5                                                                                                                                                                                                                                                           left trash cart in middle of driveway instead of at the backdoor pickup spot
7                                                                                                                                                                                                                                                                                           Trash out on time, miss again Tuesday. ALLEY
11                                                                                                                                                                                                                                                                                                            Missed- 4th week in a row.
13           

In [29]:
data_load.loc[data_load['Request '] == 'Damage to Property', 'Description']

6                                                                                                                    Trash/emptied Wednesday & now metal black-mailbox damage. \r\ncustomer wants call back and fix and replace.
173                                                                                                                                                                                 truck is cutting into yard and damaging lawn
257                                                                                                 cable lines pulled from house - caused damage to roof has pictures\r\ncomcast on site repair line itself but roof has damage
360      Customer does not understands why trash-truck bent mail-box again today. Damage mail-box won't close mail will get wet if it rains. {Repair/Please} 2nd time if any questions call her. \r\n615-876-9325\r\nJudy Cooper
384                                                                                                 

### Filter for missed pickups
To analyze the appropriate requests, I will filter the data for rows where the request is 'Missed Pickup' or the description contains either miss/missed/missing or 'no pick up'/'not picked up'

In [40]:
data_load['Description'] = data_load['Description'].fillna('[No description]')

In [129]:
missed_pickups = data_load.loc[
    (data_load['Request '].str.contains('Missed')) | 
    ## any capitalization of 'miss'
    (data_load['Description'].str.contains(r'\b[mM][iI][sS]{2}.?', regex = True)) | 
    ## any capitalization of 'no pickup'/'no pick up'/'not picked up'
    (data_load['Description'].str.contains(r'\b[nN][oO].*\b[pP][iI][cC][kK].*[uU][pP]', regex = True))].reset_index(drop = True)

In [130]:
missed_pickups

,Request Number,Date Opened,Request,Description,Incident Address,Zip Code,Trash Hauler,Trash Route,Council District,State Plan X,State Plan Y
0,25274,11/1/2017,Trash - Curbside/Alley Missed Pickup,Curb/Trash miss Tuesday.,4028 Clarksville Pike,37218.0,RED RIVER,4202,1.0,1721259.366,685444.7996
1,25276,11/1/2017,Trash - Curbside/Alley Missed Pickup,Curb/trash miss Tuesday.,6528 Thunderbird Dr,37209.0,RED RIVER,4205,20.0,1707026.753,659887.4716
2,25307,11/1/2017,Trash - Curbside/Alley Missed Pickup,missed,2603 old matthews rd,37207.0,WASTE IND,2206,2.0,1735691.771,685027.2459
3,25312,11/1/2017,Trash - Curbside/Alley Missed Pickup,Missed the even side of the road.,604 croley dr,37209.0,RED RIVER,4203,20.0,1710185.772,664205.1011
4,25327,11/1/2017,Trash Collection Complaint,"Trash out on time, miss again Tuesday. ALLEY",1816 Jo Johnston Ave,37203.0,METRO,9208,21.0,1731459.367,666013.6012
...,...,...,...,...,...,...,...,...,...,...,...
18164,267125,11/1/2019,Trash - Curbside/Alley Missed Pickup,MISSED...NEIGHBORS MISSED,2731 Murfreesboro Pike,37013.0,RED RIVER,4502,32.0,1781137.263,632448.5511
18165,267126,11/1/2019,Trash - Curbside/Alley Missed Pickup,entire alley,"1621 Long Ave, Nashville, TN 37206, United States",37206.0,METRO,9508,6.0,1749711.399,669201.6016
18166,267130,11/1/2019,Trash - Curbside/Alley Missed Pickup,missed several,"2943 Windemere Cir, Nashville, TN 37214, United States",37214.0,RED RIVER,1502,15.0,1770293.388,674936.3038
18167,267134,11/1/2019,Trash - Curbside/Alley Missed Pickup,Caller stated trash was missed & were only picked up 3x in the last 3 month.,"3325 Murfreesboro Pike, Nashville, TN 37013, United States",37013.0,RED RIVER,4502,32.0,1785224.998,627146.4002


### Normalize 'Trash Hauler'
Investigate null trash haulers - do they have route info? Can we infer?

In [131]:
missed_pickups['Trash Hauler'] = missed_pickups['Trash Hauler'].str.title()

In [132]:
missed_pickups['Trash Hauler'].value_counts(dropna = False)

Trash Hauler
Red River    13125
Metro         3098
Waste Ind     1168
NaN            778
Name: count, dtype: int64

### Normalize addresses

In [133]:
missed_pickups.loc[missed_pickups['Incident Address'].isna()]

,Request Number,Date Opened,Request,Description,Incident Address,Zip Code,Trash Hauler,Trash Route,Council District,State Plan X,State Plan Y
539,33128,12/13/2017,Trash - Curbside/Alley Missed Pickup,Missed.,NaN,NaN,NaN,NaN,NaN,NaN,NaN
761,35857,12/29/2017,Trash - Curbside/Alley Missed Pickup,daughters car parked in front,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1017,39689,1/17/2018,Trash - Curbside/Alley Missed Pickup,Trash pick up not done for Tuesday 1/16/18,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1196,41604,1/24/2018,Trash - Curbside/Alley Missed Pickup,cart still out,NaN,37218.0,Red River,3203,1.0,1715186.450,682289.9617
1646,48203,2/22/2018,Trash - Backdoor,missed for 3 weeks 698 harding pl,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2467,58920,4/6/2018,Trash - Curbside/Alley Missed Pickup,missed- trash,NaN,37206.0,Metro,9503,5.0,1747402.049,674741.0561
2508,59517,4/10/2018,Trash - Curbside/Alley Missed Pickup,Trash was not pick up last week.,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2760,62051,4/22/2018,Trash - Curbside/Alley Missed Pickup,They forgot to pick up trash from apartment complex.,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9747,157096,3/20/2019,Trash - Curbside/Alley Missed Pickup,Trash has not been picked up on the whole street.,NaN,NaN,NaN,NaN,NaN,NaN,NaN


There are only nine items missing an address - I'm just going to discard them.

In [134]:
missed_pickups = missed_pickups.loc[-missed_pickups['Incident Address'].isna()].reset_index(drop = True)

In [135]:
missed_pickups.loc[
    missed_pickups['Incident Address'].str.contains('Nashville', case = False) & 
    ~missed_pickups['Incident Address'].str.contains('Nashville')]

,Request Number,Date Opened,Request,Description,Incident Address,Zip Code,Trash Hauler,Trash Route,Council District,State Plan X,State Plan Y
1963,52415,3/8/2018,Trash - Curbside/Alley Missed Pickup,Trash has not been picked up for 3 weeks,"1246 Antioch PIke, nashville tn 37211",37211.0,NaN,NaN,13.0,1763417.519,640137.0855


There is one instance of Nashville being mentioned in all lowercase; all other mentions of it are in title case ('Nashville')

In [136]:
for i, r in missed_pickups.iterrows():
    if ('Nashville' in r['Incident Address']) or ('nashville' in r['Incident Address']):
        ## looking for any characters preceding ' Nashville' or ' nashville', followed by any number of other characters
        missed_pickups.loc[i, 'Street Address'] = re.search(r'(.*),*\s?[nN]ashville.*', r['Incident Address']).group(1).replace(',', '').replace('.', '').strip().title()
    else:
        missed_pickups.loc[i, 'Street Address'] = r['Incident Address'].replace(',', '').replace('.', '').strip().title()

In [137]:
missed_pickups.sort_values('Street Address')

,Request Number,Date Opened,Request,Description,Incident Address,Zip Code,Trash Hauler,Trash Route,Council District,State Plan X,State Plan Y,Street Address
3964,79395,6/29/2018,Trash - Curbside/Alley Missed Pickup,the entire street was missed,1 BELLE FORREST AVE C,37206.0,Metro,9502,7.0,1751942.483,677895.3105,1 Belle Forrest Ave C
3997,79884,7/2/2018,Trash - Curbside/Alley Missed Pickup,Missed entire street- carts are curbside in front of the home.,10 Belle Forrest Ave,37206.0,Metro,9502,7.0,1751718.498,678077.9341,10 Belle Forrest Ave
9636,155122,3/15/2019,Trash - Curbside/Alley Missed Pickup,MISS,"100 Bluefield Square, Nashville, TN 37214, United States",37214.0,Red River,1505,15.0,1770430.599,666861.6014,100 Bluefield Square
1947,52252,3/7/2018,Trash - Curbside/Alley Missed Pickup,Missed- trash,100 Braxton Hill Ct,37204.0,Red River,3302S,25.0,1733780.504,640909.3036,100 Braxton Hill Ct
6808,121431,12/5/2018,Trash - Curbside/Alley Missed Pickup,Missed- trash,100 Brook Hollow Rd,37205.0,Red River,1303,23.0,1708042.849,642454.6429,100 Brook Hollow Rd
...,...,...,...,...,...,...,...,...,...,...,...,...
14225,224685,8/2/2019,Trash - Curbside/Alley Missed Pickup,Trash is scheduled for collection on Thursday. Trash has not been collected for the entire street on both sides. All trash bins were at the road before 7:00am Thursday.,"Tusculum Rd, Nashville, TN , United States",37013.0,NaN,NaN,30.0,1762129.879,627839.2063,Tusculum Rd
14091,224234,8/1/2019,Trash - Curbside/Alley Missed Pickup,Trash carts were placed at the road before 7:00 a.m. Thursday. It is now 9:00 pm and trash cart has not been picked up on the entire street.,"Tusculum Rd, Nashville, TN , United States",37013.0,NaN,NaN,30.0,1762129.879,627839.2063,Tusculum Rd
13495,217242,7/18/2019,Trash - Curbside/Alley Missed Pickup,Trash was picked up on one side of the street and not the other. All trash bins were at the road before 7:00 a.m Thursday.,"Tusculum Rd, Nashville, TN , United States",37013.0,NaN,NaN,30.0,1762129.879,627839.2063,Tusculum Rd
15030,232129,8/15/2019,Trash - Curbside/Alley Missed Pickup,miss,"Westboro Dr, Nashville, TN 37209, United States",37209.0,Red River,4203,20.0,1710105.753,662979.8000,Westboro Dr


In [138]:
## beginning of input, any number of digits: r'^\d+'
missed_pickups['Building Number'] = missed_pickups['Street Address'].str.extract(r'(^\d+)')

In [139]:
missed_pickups.loc[missed_pickups['Building Number'].isna()].shape

(40, 13)

There are 40 additional rows that have no specific building listed in the address, just a street.  This is still negligible (.2% of rows) and will be discarded

In [140]:
missed_pickups = missed_pickups.loc[~missed_pickups['Building Number'].isna()].reset_index(drop = True)

In [144]:
missed_pickups['Street'] = missed_pickups['Street Address'].str.extract(r'(\b\D.*)')

In [145]:
missed_pickups

,Request Number,Date Opened,Request,Description,Incident Address,Zip Code,Trash Hauler,Trash Route,Council District,State Plan X,State Plan Y,Street Address,Building Number,Street
0,25274,11/1/2017,Trash - Curbside/Alley Missed Pickup,Curb/Trash miss Tuesday.,4028 Clarksville Pike,37218.0,Red River,4202,1.0,1721259.366,685444.7996,4028 Clarksville Pike,4028,Clarksville Pike
1,25276,11/1/2017,Trash - Curbside/Alley Missed Pickup,Curb/trash miss Tuesday.,6528 Thunderbird Dr,37209.0,Red River,4205,20.0,1707026.753,659887.4716,6528 Thunderbird Dr,6528,Thunderbird Dr
2,25307,11/1/2017,Trash - Curbside/Alley Missed Pickup,missed,2603 old matthews rd,37207.0,Waste Ind,2206,2.0,1735691.771,685027.2459,2603 Old Matthews Rd,2603,Old Matthews Rd
3,25312,11/1/2017,Trash - Curbside/Alley Missed Pickup,Missed the even side of the road.,604 croley dr,37209.0,Red River,4203,20.0,1710185.772,664205.1011,604 Croley Dr,604,Croley Dr
4,25327,11/1/2017,Trash Collection Complaint,"Trash out on time, miss again Tuesday. ALLEY",1816 Jo Johnston Ave,37203.0,Metro,9208,21.0,1731459.367,666013.6012,1816 Jo Johnston Ave,1816,Jo Johnston Ave
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18115,267125,11/1/2019,Trash - Curbside/Alley Missed Pickup,MISSED...NEIGHBORS MISSED,2731 Murfreesboro Pike,37013.0,Red River,4502,32.0,1781137.263,632448.5511,2731 Murfreesboro Pike,2731,Murfreesboro Pike
18116,267126,11/1/2019,Trash - Curbside/Alley Missed Pickup,entire alley,"1621 Long Ave, Nashville, TN 37206, United States",37206.0,Metro,9508,6.0,1749711.399,669201.6016,1621 Long Ave,1621,Long Ave
18117,267130,11/1/2019,Trash - Curbside/Alley Missed Pickup,missed several,"2943 Windemere Cir, Nashville, TN 37214, United States",37214.0,Red River,1502,15.0,1770293.388,674936.3038,2943 Windemere Cir,2943,Windemere Cir
18118,267134,11/1/2019,Trash - Curbside/Alley Missed Pickup,Caller stated trash was missed & were only picked up 3x in the last 3 month.,"3325 Murfreesboro Pike, Nashville, TN 37013, United States",37013.0,Red River,4502,32.0,1785224.998,627146.4002,3325 Murfreesboro Pike,3325,Murfreesboro Pike


#### Some trash routes are not purely numeric - they have an 'S' at the end, which feels intentional.  If we do any analysis by route, we might want to look into that.

In [148]:
missed_pickups['Trash Route'].value_counts(dropna = False)

Trash Route
NaN      769
4504     334
3302     297
4404     257
1303     255
        ... 
4504S      3
2405S      3
3303S      2
2505S      2
1502S      1
Name: count, Length: 173, dtype: int64

In [149]:
missed_pickups.sort_values('Trash Route')['Trash Route'].unique()

array(['1201', '1202', '1202S', '1203', '1204', '1205', '1301', '1302',
       '1302S', '1303', '1303S', '1304', '1305', '1308', '1309', '1401',
       '1402', '1403', '1404', '1405', '1501', '1502', '1502S', '1503',
       '1504', '1504S', '1505', '1507', '2201', '2202', '2203', '2204',
       '2205', '2206', '2207', '2301', '2301S', '2302', '2303', '2303S',
       '2304', '2304S', '2305', '2305S', '2306', '2308', '2401', '2402',
       '2402S', '2403', '2404', '2405', '2405S', '2408', '2501', '2502',
       '2503', '2504', '2505', '2505S', '2506', '2509', '3201', '3202',
       '3203', '3204', '3205', '3206', '3207', '3208', '3212', '3214',
       '3301', '3301S', '3302', '3302S', '3303', '3303S', '3304', '3304S',
       '3305', '3305S', '3306', '3307', '3312', '3314', '3401', '3402',
       '3402S', '3403', '3404', '3405', '3406', '3407', '3411', '3412',
       '3414', '3501', '3502', '3503', '3504', '3505', '3512', '3514',
       '4201', '4201S', '4202', '4203', '4203S', '4204', '4

## Determine damages due to missed pickups
* The first missed pickup at an address will not result in a fine
* Every subsequent missed pickup will result in a $200 fine

For each trash hauler, we need:  
* A: A list of each unique addresses with missed pickup
* B: The number of unique addresses with missed pickups
* C: The number of times that each address was missed
  
Then the fine should be 200 * (SUM(C) - B)

In [150]:
unique_haulers = missed_pickups['Trash Hauler'].unique()

In [169]:
damages = []

for hauler in unique_haulers:
    if str(hauler) == 'nan':
        missed = missed_pickups.loc[missed_pickups['Trash Hauler'].isna()]
    else:
        missed = missed_pickups.loc[missed_pickups['Trash Hauler'] == hauler]

    rows = missed.shape[0]
    addresses = missed['Street Address'].nunique()
    fine = 200 * (rows - addresses)

    damages.append({'hauler': hauler, 'fine': fine})

damages = pd.DataFrame(damages)
damages

,hauler,fine
0,Red River,913400
1,Waste Ind,77200
2,Metro,215000
3,NaN,16000


#### Let's refine this solution:
* Ask whether the fine should only be for *consecutive* missed pickups
* Review the data to make sure there aren't multiple calls in for addresses on the same day